In [1]:
from datasets import load_dataset

ds = load_dataset("cnamuangtoun/resume-job-description-fit")

c:\Users\andri_c\Desktop\IA\CV-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['resume_text', 'job_description_text', 'label'],
        num_rows: 6241
    })
    test: Dataset({
        features: ['resume_text', 'job_description_text', 'label'],
        num_rows: 1759
    })
})


In [3]:
print(ds["train"][0])

{'resume_text': "SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.\nHighlights-Soft Skills: Public Speaking, Public Relations, Team Building, Project Management, Procedure writing, Staff Supervision and Management, Ability to interface with professionals on all levels. Accomplishments, Honors, and Activities -Board of Directors Member for the Food Bank of Corpus Christi from November 2010 to April 2013. -Held Life Insurance License -Basketball Official (Referee) High School Varsity Level.\nExperienceAccountant,08/2014-05/2015Aspirus–Owen,WI,Perform daily and routine accounting functions for two main companies and five small royalty companies. Responsibilities include but are not limited to the following: Accounts Payable, Accounts Receivable, Manage and reconcile funds for multiple banks accou

In [4]:
print(set(ds["train"]["label"]))

{'Potential Fit', 'No Fit', 'Good Fit'}


In [5]:
from sentence_transformers import InputExample

label_map = {"No Fit": 0.0, "Potential Fit": 0.5, "Good Fit": 1.0}

train_examples = [
    InputExample(
        texts=[row["resume_text"], row["job_description_text"]],
        label=label_map[row["label"]]
    )
    for row in ds["train"]
]

print(len(train_examples))
print(train_examples[0])

6241
<InputExample> label: 0.0, texts: SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.
Highlights-Soft Skills: Public Speaking, Public Relations, Team Building, Project Management, Procedure writing, Staff Supervision and Management, Ability to interface with professionals on all levels. Accomplishments, Honors, and Activities -Board of Directors Member for the Food Bank of Corpus Christi from November 2010 to April 2013. -Held Life Insurance License -Basketball Official (Referee) High School Varsity Level.
ExperienceAccountant,08/2014-05/2015Aspirus–Owen,WI,Perform daily and routine accounting functions for two main companies and five small royalty companies. Responsibilities include but are not limited to the following: Accounts Payable, Accounts Receivable, Manage and reconcile funds for 

In [7]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# On reconstruit un Dataset avec les bonnes colonnes,
# à partir des 100 premiers train_examples déjà créés
train_dataset = Dataset.from_dict({
    "sentence1": [ex.texts[0] for ex in train_examples[:100]],
    "sentence2": [ex.texts[1] for ex in train_examples[:100]],
    "score": [ex.label for ex in train_examples[:100]],
})

train_loss = losses.CosineSimilarityLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_dataset,
    loss=train_loss,
)

trainer.train()

model.save("models/embedding_model_test")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3249.51it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.
c:\Users\andri_c\Desktop\IA\CV-AI\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.04it/s]


In [ ]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=100,
    output_path="models/embedding_model"
)